# Start coding and Enjoy the journey
> Notebook adattato per essere eseguito dalla **root del progetto**.

## Setup: Install dependencies & Download Datasets

Esegue `setup_deps.sh` dalla root — installa le dipendenze e scarica i dataset in un solo passaggio.

In [ ]:
!bash setup_deps.sh

## Run your First VPR Evaluation

Imposta `DATASET` con uno dei dataset disponibili: `sf_xs`, `tokyo_xs`, `gsv_xs`, `robotcar_cut_2m_formatted`.

Le variabili `DATABASE_FOLDER` e `QUERIES_FOLDER` vengono costruite automaticamente.

In [ ]:
DATASET = 'sf_xs'           # <-- cambia qui: sf_xs | tokyo_xs | gsv_xs | robotcar_cut_2m_formatted
SPLIT   = 'test'             # <-- train | val | test (secondo quanto disponibile per il dataset)

DATABASE_FOLDER = f'data/{DATASET}/{SPLIT}/database'
QUERIES_FOLDER  = f'data/{DATASET}/{SPLIT}/queries'
LOG_DIR         = f'log_dir/{DATASET}'

print(f'Database : {DATABASE_FOLDER}')
print(f'Queries  : {QUERIES_FOLDER}')
print(f'Log dir  : {LOG_DIR}')

In [ ]:
!yes | python deps/VPR-methods-evaluation/main.py \
    --num_workers 8 \
    --batch_size 32 \
    --log_dir {LOG_DIR} \
    --method=cosplace --backbone=ResNet18 --descriptors_dimension=512 \
    --image_size 512 512 \
    --database_folder {DATABASE_FOLDER} \
    --queries_folder  {QUERIES_FOLDER} \
    --num_preds_to_save 20 \
    --recall_values 1 5 10 20 \
    --save_for_uncertainty

## Run Image Matching on Retrieval Results

In [ ]:
PREDS_DIR = f'{LOG_DIR}/preds'   # adatta se main.py salva altrove

!python deps/match_queries_preds.py \
    --preds-dir {PREDS_DIR} \
    --matcher 'superpoint-lg' \
    --device 'cuda' \
    --num-preds 20

## Check Re-ranking Performance

In [ ]:
INLIERS_DIR = f'{LOG_DIR}/inliers'   # adatta se match_queries_preds.py salva altrove

!python deps/reranking.py \
    --preds-dir   {PREDS_DIR} \
    --inliers-dir {INLIERS_DIR} \
    --num-preds 20 \
    --recall-values 1 5 10 20

## Perform Uncertainty Evaluation

In [ ]:
Z_DATA_PATH = f'{LOG_DIR}/z_data.pkl'   # adatta al path effettivo del file z-data

!python -m deps.vpr_uncertainty.eval \
    --preds-dir   {PREDS_DIR} \
    --inliers-dir {INLIERS_DIR} \
    --z-data-path {Z_DATA_PATH}